# Phase 2 QRC all-pairs ZZ readout probe

This notebook turns the all-pairs ZZ readout script into a more inspectable workflow.

Goal: test whether the full-topology TFIM reservoir contains useful long-range pairwise correlation information that was being discarded by nearest-neighbor-only ZZ readout.

The key controlled comparison is:

- same data split
- same PCA-6 preprocessing
- same 40-day windows
- same leaky input integration
- same 6-qubit full-topology TFIM reservoir
- same anchors, Trotter steps, virtual nodes, ridge readout
- changed readout only: nearest-neighbor ZZ vs all-pairs ZZ

Use this notebook to inspect whether the extra long-range ZZ observables help, merely add noise, or summon overfitting goblins.


## 1. Setup

Run from the repository root if possible. If the notebook is opened from `notebooks/`, the path patch below should still work.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Make repo-root imports work whether the notebook is launched from repo root or notebooks/.
ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT))

OUT_DIR = ROOT / 'results' / 'tables'
OUT_DIR.mkdir(parents=True, exist_ok=True)

ROOT


In [ ]:
from qpitome_qrc.data.features import FEATURE_COLUMNS
from qpitome_qrc.data.loaders import load_phase2_volatility_data
from qpitome_qrc.data.pca import fit_transform_pca_splits_train_only
from qpitome_qrc.data.splits import chronological_tabular_split
from qpitome_qrc.evaluation.metrics import evaluate_volatility_forecast
from qpitome_qrc.qrc.tfim_reservoir import TFIMQRCConfig

# Helper functions are kept in the script so notebook and script use the same logic.
from scripts.run_phase2_qrc_all_pairs_zz_probe import (
    build_feature_matrix_with_zz_mode,
    expanded_feature_names,
    fit_readout,
    high_vol_metrics,
    leaky_integrate_windows,
    make_qrc_sequence_splits,
    summarize_feature_selection,
)


## 2. Load data and reproduce the final QRC preprocessing

This uses the same target and PCA-6 setup as the final encoding/readout QRC run.


In [ ]:
target = 'future_rv_20d'

df = load_phase2_volatility_data()
splits = chronological_tabular_split(df)

pca6 = fit_transform_pca_splits_train_only(
    splits,
    feature_columns=FEATURE_COLUMNS,
    target_columns=[target],
    n_components=6,
    prefix='pca6',
)

raw_seq = make_qrc_sequence_splits(
    pca6.splits,
    feature_columns=pca6.feature_columns,
    target_column=target,
    lookback_days=40,
)

leaky_seq = {
    split: (leaky_integrate_windows(X, leak=0.3), y, dates)
    for split, (X, y, dates) in raw_seq.items()
}

X_train, y_train, train_dates = leaky_seq['train']
X_val, y_val, val_dates = leaky_seq['val']
X_test, y_test, test_dates = leaky_seq['test']

{k: (v[0].shape, v[1].shape) for k, v in leaky_seq.items()}


## 3. Final QRC configuration

This is the final Phase 2 QRC configuration. The only experimental variable below will be the ZZ readout mode.


In [ ]:
config = TFIMQRCConfig(
    qubits=6,
    pca_components=6,
    lookback_days=40,
    anchor_count=10,
    anchor_policy='recent',
    observable_mode='zxzz',
    collect_anchor_features=True,
    topology='full',
    trotter_steps_per_anchor=3,
    virtual_nodes_per_anchor=3,
    coupling_scale=0.7,
    transverse_field=0.5,
    evolution_time=0.5,
    angle_max=np.pi / 2,
    ridge_alpha=1000.0,
    target_transform='log',
    seed=42,
    use_disorder=True,
    disorder_strength=0.20,
)
config


## 4. Build nearest-neighbor and all-pairs ZZ features

This is the slow part. The notebook caches the raw feature matrices to `.npz` so repeated readout sweeps do not recompute the reservoir.

Feature counts expected:

- nearest ZZ: 6 Z + 6 X + 5 ZZ = 17 observables per virtual node; 30 blocks → 510? Depending on selected recent anchors, actual block count may be lower if anchor selection returns unique indices.
- all-pairs ZZ: 6 Z + 6 X + 15 ZZ = 27 observables per virtual node.

The empirical feature counts below are what matter.


In [ ]:
CACHE = OUT_DIR / 'phase2_qrc_all_pairs_zz_raw_features_cache.npz'

if CACHE.exists():
    print(f'Loading cache: {CACHE}')
    cache = np.load(CACHE)
    H_train_nearest = cache['H_train_nearest']
    H_val_nearest = cache['H_val_nearest']
    H_test_nearest = cache['H_test_nearest']
    H_train_all = cache['H_train_all']
    H_val_all = cache['H_val_all']
    H_test_all = cache['H_test_all']
else:
    H_train_nearest = build_feature_matrix_with_zz_mode(X_train, config, zz_mode='nearest', verbose=True)
    H_val_nearest = build_feature_matrix_with_zz_mode(X_val, config, zz_mode='nearest', verbose=True)
    H_test_nearest = build_feature_matrix_with_zz_mode(X_test, config, zz_mode='nearest', verbose=True)

    H_train_all = build_feature_matrix_with_zz_mode(X_train, config, zz_mode='all', verbose=True)
    H_val_all = build_feature_matrix_with_zz_mode(X_val, config, zz_mode='all', verbose=True)
    H_test_all = build_feature_matrix_with_zz_mode(X_test, config, zz_mode='all', verbose=True)

    np.savez_compressed(
        CACHE,
        H_train_nearest=H_train_nearest,
        H_val_nearest=H_val_nearest,
        H_test_nearest=H_test_nearest,
        H_train_all=H_train_all,
        H_val_all=H_val_all,
        H_test_all=H_test_all,
    )
    print(f'Saved cache: {CACHE}')

{
    'nearest': (H_train_nearest.shape, H_val_nearest.shape, H_test_nearest.shape),
    'all': (H_train_all.shape, H_val_all.shape, H_test_all.shape),
}


## 5. Controlled nearest vs all-pairs comparison

This reproduces the script result with top-k = 240 and ridge alpha = 1000.


In [ ]:
def evaluate_run(run_name, zz_mode, H_train, H_val, H_test, top_k=240, alpha=1000.0):
    pred_train, pred_val, pred_test, selected_idx = fit_readout(
        H_train, H_val, H_test, y_train, top_k=top_k, alpha=alpha
    )

    rows = []
    for split_name, y, pred in [
        ('train', y_train, pred_train),
        ('val', y_val, pred_val),
        ('test', y_test, pred_test),
    ]:
        m = evaluate_volatility_forecast(y, pred)
        rows.append({
            'run_name': run_name,
            'zz_mode': zz_mode,
            'top_k': top_k,
            'alpha': alpha,
            'split': split_name,
            'rmse': m.rmse,
            'qlike': m.qlike,
            'mz_r2': m.mz_r2,
            'corr': float(np.corrcoef(y, pred)[0, 1]),
            'pred_std': float(np.std(pred)),
        })

    names = expanded_feature_names(config, zz_mode=zz_mode)
    sel = summarize_feature_selection(selected_idx, names)
    sel.update({'run_name': run_name, 'zz_mode': zz_mode, 'top_k': top_k, 'alpha': alpha, 'n_raw_features': H_train.shape[1]})

    preds = {
        'train': pred_train,
        'val': pred_val,
        'test': pred_test,
    }
    return pd.DataFrame(rows), sel, preds

metrics_nearest, sel_nearest, preds_nearest = evaluate_run(
    'linear_clip_top240_alpha1000_zz_nearest',
    'nearest',
    H_train_nearest, H_val_nearest, H_test_nearest,
)
metrics_all, sel_all, preds_all = evaluate_run(
    'linear_clip_top240_alpha1000_zz_all',
    'all',
    H_train_all, H_val_all, H_test_all,
)

metrics_compare = pd.concat([metrics_nearest, metrics_all], ignore_index=True)
selection_compare = pd.DataFrame([sel_nearest, sel_all])

metrics_compare


In [ ]:
selection_compare


In [ ]:
test_compare = metrics_compare.query("split == 'test'").copy()
test_compare[['run_name', 'rmse', 'qlike', 'mz_r2', 'corr', 'pred_std']]


## 6. High-volatility warning diagnostics

Use train-defined actual volatility thresholds. q80 is the watch-level layer; q90/q95 are harder tail diagnostics.


In [ ]:
thresholds = {f'q{q}': float(np.quantile(y_train, q / 100.0)) for q in [80, 90, 95]}

def high_vol_table(run_name, zz_mode, preds):
    rows = []
    for split_name, y, pred in [
        ('train', y_train, preds['train']),
        ('val', y_val, preds['val']),
        ('test', y_test, preds['test']),
    ]:
        for q_name, threshold in thresholds.items():
            row = high_vol_metrics(y, pred, threshold)
            row.update({
                'run_name': run_name,
                'zz_mode': zz_mode,
                'split': split_name,
                'quantile': q_name,
                'threshold': threshold,
            })
            rows.append(row)
    return pd.DataFrame(rows)

high_vol_compare = pd.concat([
    high_vol_table('linear_clip_top240_alpha1000_zz_nearest', 'nearest', preds_nearest),
    high_vol_table('linear_clip_top240_alpha1000_zz_all', 'all', preds_all),
], ignore_index=True)

high_vol_compare.query("split == 'test'").sort_values(['quantile', 'zz_mode'])


## 7. Goblin diagnostics

These quick checks look for suspicious behavior:

- train improves sharply while validation/test degrade
- prediction standard deviation explodes
- q90/q95 fires randomly
- long-range ZZ features dominate without improving validation/test


In [ ]:
pivot = metrics_compare.pivot_table(index=['zz_mode'], columns='split', values=['rmse', 'mz_r2', 'corr', 'pred_std'])
pivot


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for mode, preds in [('nearest', preds_nearest), ('all', preds_all)]:
    ax.plot(pd.to_datetime(test_dates), preds['test'], label=f'{mode} ZZ prediction', alpha=0.8)
ax.plot(pd.to_datetime(test_dates), y_test, label='actual future RV 20d', alpha=0.7)
ax.set_title('Nearest vs all-pairs ZZ test predictions')
ax.set_ylabel('future_rv_20d')
ax.legend()
fig.autofmt_xdate()
plt.show()


## 8. Optional: top-k / alpha readout sweep for all-pairs ZZ

This is the natural follow-up because all-pairs ZZ increases the feature space. We test whether the readout wants more or fewer selected features and stronger/weaker ridge regularization.

Keep this disciplined. The question is not `can we tune forever?` The question is whether the all-pairs feature space gives a clear, stable improvement without goblins.


In [ ]:
top_k_grid = [120, 240, 360, 480, H_train_all.shape[1]]
alpha_grid = [300.0, 1000.0, 3000.0, 10000.0]

sweep_metrics = []
sweep_selection = []

for top_k in top_k_grid:
    for alpha in alpha_grid:
        run_name = f'all_pairs_zz_top{top_k}_alpha{alpha:g}'
        m_df, sel, _preds = evaluate_run(
            run_name,
            'all',
            H_train_all, H_val_all, H_test_all,
            top_k=top_k,
            alpha=alpha,
        )
        sweep_metrics.append(m_df)
        sweep_selection.append(sel)

sweep_metrics = pd.concat(sweep_metrics, ignore_index=True)
sweep_selection = pd.DataFrame(sweep_selection)

sweep_metrics.to_csv(OUT_DIR / 'phase2_qrc_all_pairs_zz_readout_sweep.csv', index=False)
sweep_selection.to_csv(OUT_DIR / 'phase2_qrc_all_pairs_zz_readout_sweep_feature_selection.csv', index=False)

sweep_metrics.head()


In [ ]:
# Sort by validation RMSE first to avoid choosing a test-only mirage.
val_rank = (
    sweep_metrics.query("split == 'val'")
    .sort_values(['rmse', 'qlike'], ascending=[True, True])
    [['run_name', 'top_k', 'alpha', 'rmse', 'qlike', 'mz_r2', 'corr', 'pred_std']]
)
val_rank.head(10)


In [ ]:
# Inspect corresponding test scores for the top validation candidates.
top_val_names = val_rank.head(10)['run_name'].tolist()
sweep_metrics.query("split == 'test' and run_name in @top_val_names")
    .sort_values('run_name')
    [['run_name', 'top_k', 'alpha', 'rmse', 'qlike', 'mz_r2', 'corr', 'pred_std']]


In [ ]:
# Test-only ranking is useful, but do not select from this alone unless validation is also sane.
test_rank = (
    sweep_metrics.query("split == 'test'")
    .sort_values(['rmse', 'qlike'], ascending=[True, True])
    [['run_name', 'top_k', 'alpha', 'rmse', 'qlike', 'mz_r2', 'corr', 'pred_std']]
)
test_rank.head(10)


## 9. Paper-ready interpretation cell

Fill this in after inspecting the tables. Keep it honest.


In [ ]:
nearest_test = test_compare.set_index('zz_mode').loc['nearest']
all_test = test_compare.set_index('zz_mode').loc['all']
sel_all = selection_compare.set_index('zz_mode').loc['all']

paper_sentence = (
    f"Because the final TFIM reservoir uses full-topology interactions, we tested whether the readout should include all pairwise ZZ correlations rather than only nearest-neighbor ZZ terms. "
    f"The all-pairs readout increased the raw feature count to {int(sel_all['n_raw_features'])} and selected {int(sel_all['selected_long_range_zz_features'])} long-range ZZ correlators among the top {int(sel_all['selected_features'])} train-selected features. "
    f"Relative to nearest-neighbor ZZ, test RMSE changed from {nearest_test['rmse']:.6f} to {all_test['rmse']:.6f}, QLIKE from {nearest_test['qlike']:.6f} to {all_test['qlike']:.6f}, and MZ R² from {nearest_test['mz_r2']:.6f} to {all_test['mz_r2']:.6f}. "
    f"This suggests that long-range reservoir correlations contain useful volatility-transition information, although the improvement remains modest and high-tail calibration remains Phase 3 work."
)
print(paper_sentence)


## 10. Save compact comparison tables

These files can be used directly in the paper/draft update.


In [ ]:
metrics_compare.to_csv(OUT_DIR / 'phase2_qrc_all_pairs_zz_probe_metrics_notebook.csv', index=False)
high_vol_compare.to_csv(OUT_DIR / 'phase2_qrc_all_pairs_zz_probe_high_vol_notebook.csv', index=False)
selection_compare.to_csv(OUT_DIR / 'phase2_qrc_all_pairs_zz_probe_feature_selection_notebook.csv', index=False)

pd.concat([
    pd.DataFrame({
        'date': pd.to_datetime(test_dates),
        'actual_future_rv_20d': y_test,
        'qrc_pred_future_rv_20d': preds_nearest['test'],
        'run_name': 'linear_clip_top240_alpha1000_zz_nearest',
        'zz_mode': 'nearest',
    }),
    pd.DataFrame({
        'date': pd.to_datetime(test_dates),
        'actual_future_rv_20d': y_test,
        'qrc_pred_future_rv_20d': preds_all['test'],
        'run_name': 'linear_clip_top240_alpha1000_zz_all',
        'zz_mode': 'all',
    }),
], ignore_index=True).to_csv(OUT_DIR / 'phase2_qrc_all_pairs_zz_probe_predictions_notebook.csv', index=False)

print('Saved notebook comparison tables to', OUT_DIR)
